In [1]:
import os
import urllib.request
import urllib.error
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
llm = ChatOpenAI(
    base_url=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    timeout=300,
    temperature=0.5,
    max_tokens=25000
)

In [3]:
llm.invoke("What is the capital of South Korea?").content

'The capital of South Korea is Seoul.'

In [4]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
)

In [5]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': 'I can’t access real-time data in this chat. If you’d like, I can help you quickly check it yourself or convert an amount once you have the rate.\n\nHow to get the current KRW to USD rate:\n- Google search: “KRW to USD” (shows the current rate at the top)\n- Financial sites: XE, OANDA, Yahoo Finance, Reuters\n- Official: Bank of Korea publishes daily rates (use the mid-market rate)\n- Your bank or card issuer may show a different rate for transfers or purchases\n\nHow to convert once you have a rate:\n- If 1 USD = X KRW and you have Y KRW, then USD = Y / X\n- Example: If 1 USD = 1,300 KRW and you have 10,000 KRW, then USD ≈ 10,000 / 1,300 ≈ 7.69\n\nIf you share the amount in KRW you want to convert and the current rate you see, I’ll calculate it for you. Also keep in mind that the rate you get from a bank or card can differ from the mid-market rate.'}]


In [6]:
def bfmsgs(result):
    """Extract and display text content from agent result messages."""
    text = "\n\n".join(
        block["text"] for block in result["messages"][-1].content_blocks
        if block.get("type") == "text"
    )
    display(Markdown(text))

In [7]:
bfmsgs(result)

I can’t access real-time data in this chat. If you’d like, I can help you quickly check it yourself or convert an amount once you have the rate.

How to get the current KRW to USD rate:
- Google search: “KRW to USD” (shows the current rate at the top)
- Financial sites: XE, OANDA, Yahoo Finance, Reuters
- Official: Bank of Korea publishes daily rates (use the mid-market rate)
- Your bank or card issuer may show a different rate for transfers or purchases

How to convert once you have a rate:
- If 1 USD = X KRW and you have Y KRW, then USD = Y / X
- Example: If 1 USD = 1,300 KRW and you have 10,000 KRW, then USD ≈ 10,000 / 1,300 ≈ 7.69

If you share the amount in KRW you want to convert and the current rate you see, I’ll calculate it for you. Also keep in mind that the rate you get from a bank or card can differ from the mid-market rate.

In [8]:
def get_fx(currency: str) -> str:
    """Get exchange rate for a given currency."""
    if currency.upper() in ["KRW", "USD"]:
        return "1500"

In [9]:
agent = create_agent(
    model=llm,
    system_prompt="You are a helpful assistant",
    tools=[get_fx]
)

In [10]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's KRW worth in USD right now?"}]}
)
bfmsgs(result)

Approximate rate: 1 KRW ≈ 0.0006667 USD (based on 1 USD ≈ 1,500 KRW).

So:
- 1 KRW ≈ 0.000667 USD
- 10,000 KRW ≈ 6.67 USD

Rates change continuously. Want me to convert a specific KRW amount?

In [11]:
# @tool # Uncomment this line to register the function as a LangChain tool
def fetch_text_from_url(url: str) -> str:
    """Fetch the document from a URL and extract text from div id='wrapper'."""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"

    text = raw.decode("utf-8", errors="replace")
    soup = BeautifulSoup(text, "html.parser")
    wrapper = soup.find("div", id="wrapper")

    if wrapper:
        return wrapper.get_text(strip=True)
    return "Wrapper div not found"

In [12]:
fetch_text_from_url("https://www.hanyang.ac.kr/re12")

'LanguageThis provides an unofficial translation by the Google Translate. Information on International Admissions, Research, and Support is available on ourEnglish website검색#ABC#ABC#ABCENGHY-in기부MENUHY-in공지학부모기부사이트맵한양소개서울캠퍼스ERICA캠퍼스입학/교육연구/산학/창업한양 광장한양대는총장/경영상징과 UI설립/역사대학현황동문/발전기금한양대학교는건학 정신대학 규정 (학칙)Vision 2030윤리헌장조직도한양학원브랜드 아카이브대학평의원회인사말약력과 경력2026 중기발전계획총장 활동연설문역대 총장총장에게 바란다부총장/처장단한양의 상징교가UIUI 활용색상 및 서체한양 캐릭터슬로건설립자 메시지+연보한양의 역사최초/유일대학 역사관백서한눈에 보는 한양통계 연보대학 요람해외 자매결연 대학자체평가한양 지속가능경영한양 SDGs대학정보공시대외협력처 바로가기한양 동문발전기금 리포트발전기금 뉴스레터캠퍼스 소개학사 안내행정 지원 안내기관/단체대학 생활서울캠퍼스는대학/학과 소개대학원캠퍼스 지도+찾아오는길주차안내한양둘레길(8경)전화번호 안내학사일정수업 안내온라인 강좌전공제도학적변동성적졸업학점인정계절학기학점 교류업무별 담당 부서 안내등록 안내장학 안내증명발급 안내기숙사 안내교직이수교원자격증발급학생대관외부대관통합 양식자료실행정기관부속/부설기관학생 기구동아리 안내학생 언론사2025 캠퍼스 가이드북오늘의 메뉴IT 서비스복지 매장복지 혜택보건 안내의료혜택학생 상담인권센터학생증/국제학생증병무 안내고시반 안내캠퍼스 소개학사 안내행정 지원 안내기관/단체대학 생활ERICA캠퍼스는대학/학과 소개대학원연혁캠퍼스 지도+찾아오는길통학·셔틀버스주차안내ERICA 정부재정지원사업전화번호 안내학사일정수업 안내온라인 강좌전공제도학적변동성적졸업학점인정계절학기학점 교류업무별 담당 부서 안내등록 안내장학 안내기숙사 안내증명발급 안내교직 이수대관 안내통합 양식자

In [13]:
language = "English"
SYSTEM_PROMPT = f"""You are a menu finder assistant. You can find menu information and prices from restaurant websites in Korea and provide it to users. Always answer in {language} and use the following tools below to fetch menu information.

## Capabilities

- `fetch_text_from_url`: loads website text from a URL into the conversation. Be aware of the structure of a website and extract only relevant information."""

In [14]:
agent = create_agent(
    model=llm,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT
)

In [15]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's on the menu today at Hanyang University? Find it at https://www.hanyang.ac.kr/re12"}]}
)
bfmsgs(result)

Here’s what the HYU re12 page shows as 오늘의 메뉴 (Today’s Menu) for both campuses, based on the page dated 2026-04-30 (Fri). Note that only some items have prices listed in this view; other items appear without prices on the page.

Seoul Campus (서울캠퍼스) – today’s menu
- 조식 (Brunch / 천원의 아침밥)
  - Brunch Set (토스트/미니꿀호떡빵) with 양배추샐러드&드레싱, 크림스프, 커피캔, 후랑크, 케찹볶음, 단호박범벅, 양념밥
  - Price: 1,000원
- 중식
  - 유부주머니우동 (Fried Tofu Pocket Udon) with 김가루양념밥, 양배추샐러드&드레싱, 배추김치
  - Price: 4,500원
  - 도쿄함박스테이크&계란후라이 (Hamburger Steak & Fried Egg) with 쌀밥, 샐러드, 파스타, 해시브라운, 오이피클, 배추김치
  - Price: 6,500원
  - 스파이시치킨스테이크 (Spicy Chicken Steak) with 쌀밥, 샐러드, 파스타, 해시브라운, 오이피클, 배추김치
  - Price: 7,500원
- Other 조식/중식/석식 items listed on the page include various Korean/Japanese-inspired dishes (e.g., 꼬치어묵탕, 토스트/카스테라 Brunch Set, 닭고기김치볶음, 가츠동, 규동, 훈제오리블랙빈볶음밥, 제주식고기국수, 유부주머니우동, 도쿄함박스테이크, 스파이시치킨스테이크, etc.). These are shown in the page excerpt but prices are not provided in this view.

ERICA Campus (ERICA캠퍼스) – today’s menu
- The ERICA page presents a similar set of 메뉴 items (조식 and 중식) with the same structure and dish names as listed above, including Brunch Set items and the same menu translations. Prices shown in the excerpt are the same as Seoul Campus for the items that are labeled with prices (1,000원 for Brunch Set; 4,500원; 6,500원; 7,500원 for the first few lunch items). Other items on the ERICA page are listed without explicit prices in this view.

Quick tips
- The page is labeled “2026/04/30 금주의 메뉴 오늘의 메뉴” for both campuses, so the exact items can vary day to day.
- If you want every item’s price clearly, or the full, exact today’s list, I can pull the page again and extract price details item by item.

Would you like me to pull and format the complete today’s menu (with any missing prices clarified) for Seoul and ERICA campuses, or help navigate to the “오늘의 메뉴” sections on the site?